# **Federated Tree-Based Model for Financial Risk Analysis**


**Objective**

Implement Federated Learning using Tree-Based Models for credit risk prediction.

**Key concepts:**

1. Local training at multiple clients

2. No raw data sharing

3. Aggregation at central server

4. Used for financial risk / credit scoring

**Step 1 — Install Libraries**

In [ ]:
!pip install scikit-learn pandas numpy matplotlib

**Step 2 — Import Libraries**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import random

**Step 3 — Create Financial Risk Dataset**

Simulated features:

1. income
2. loan amount
3. credit score
4. age
5. repayment history

In [ ]:
np.random.seed(42)

data_size = 1000

data = pd.DataFrame({

    "income": np.random.randint(20000,120000,data_size),
    "loan_amount": np.random.randint(1000,50000,data_size),
    "credit_score": np.random.randint(300,850,data_size),
    "age": np.random.randint(21,65,data_size),
    "repayment_history": np.random.randint(0,2,data_size)
})

# Risk label
data["risk"] = (
    (data["credit_score"] < 500) |
    (data["repayment_history"] == 0)
).astype(int)

data.head()

,income,loan_amount,credit_score,age,repayment_history,risk
0,35795,49447,334,61,1,1
1,20860,35529,744,59,1,0
2,96820,11167,392,33,1,1
3,74886,11554,506,31,0,1
4,26265,18260,652,39,0,1


Risk label:

1 → High Risk

0 → Low Risk

**Step 4 — Split Dataset**

In [ ]:

X = data.drop("risk",axis=1)
y = data["risk"]

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

**Step 5 — Simulate Federated Clients (Banks)**

In [ ]:
NUM_CLIENTS = 5

client_data = []

size = len(X_train)//NUM_CLIENTS

for i in range(NUM_CLIENTS):

    start = i*size
    end = start+size

    X_client = X_train.iloc[start:end]
    y_client = y_train.iloc[start:end]

    client_data.append((X_client,y_client))

Each client acts like a separate financial institution.

Step 6 — Local Tree Model Training **bold text**

In [ ]:
def train_client_tree(X,y):

    model = RandomForestClassifier(
        n_estimators=50,
        max_depth=5,
        random_state=42
    )

    model.fit(X,y)

    return model

**Step 7 — Federated Aggregation**

Tree models cannot be averaged like neural networks.

Instead we use prediction aggregation (ensemble voting).

In [ ]:
def federated_predict(models,X):

    predictions = []

    for model in models:
        pred = model.predict(X)
        predictions.append(pred)

    predictions = np.array(predictions)

    final_prediction = np.round(predictions.mean(axis=0))

    return final_prediction

**Step 8 — Federated Training**

In [ ]:
client_models = []

for i in range(NUM_CLIENTS):

    X_client,y_client = client_data[i]

    model = train_client_tree(X_client,y_client)

    client_models.append(model)

    print("Client",i+1,"model trained")

Client 1 model trained
Client 2 model trained
Client 3 model trained
Client 4 model trained
Client 5 model trained


**Step 9 — Global Prediction**

In [ ]:
y_pred = federated_predict(client_models,X_test)

accuracy = accuracy_score(y_test,y_pred)

print("Federated Model Accuracy:",accuracy)

Federated Model Accuracy: 1.0


**Step 10 — Risk Prediction Example**

In [ ]:
sample = pd.DataFrame({

    "income":[40000],
    "loan_amount":[20000],
    "credit_score":[450],
    "age":[35],
    "repayment_history":[0]
})

prediction = federated_predict(client_models,sample)

if prediction[0] == 1:
    print("High Financial Risk")
else:
    print("Low Financial Risk")

High Financial Risk
